In [ ]:
#TD3+动作投影+残差动作+csv导出每10回合保存一次
# ==================== TD3 + 动作投影 + 残差动作 ====================e=0.5
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from collections import deque
import random
import copy
import matplotlib.pyplot as plt
import os

# ==================== 功率曲线 ====================
power_curve_68 = {3.0: 103, 3.5: 179, 4.0: 290, 4.5: 432, 5.0: 620, 5.5: 847, 6.0: 1120, 6.5: 1448,
                  7.0: 1827, 7.5: 2272, 8.0: 2777, 8.5: 3343, 9.0: 3956, 9.5: 4644, 10.0: 5394, 10.5: 6075,
                  11.0: 6763, 11.5: 6800, 12.0: 6800, 12.5: 6800, 13.0: 6800, 13.5: 6800, 14.0: 6800, 14.5: 6800,
                  15.0: 6800, 15.5: 6800, 16.0: 6800, 16.5: 6800, 17.0: 6800, 17.5: 6800, 18.0: 6800, 18.5: 6800,
                  19.0: 6800, 19.5: 6800, 20.0: 6800, 20.5: 6800, 21.0: 6800, 21.5: 6800, 22.0: 6800, 22.5: 6800,
                  23.0: 6800, 23.5: 6800, 24.0: 6800, 24.5: 6800, 25.0: 6800}

power_curve_83 = {3.0: 58, 3.5: 125, 4.0: 248, 4.5: 426, 5.0: 643, 5.5: 923, 6.0: 1260, 6.5: 1648,
                  7.0: 2105, 7.5: 2642, 8.0: 3271, 8.5: 3970, 9.0: 4686, 9.5: 5430, 10.0: 6212, 10.5: 7014,
                  11.0: 7824, 11.5: 8300, 12.0: 8300, 12.5: 8300, 13.0: 8300, 13.5: 8300, 14.0: 8300, 14.5: 8300,
                  15.0: 8300, 15.5: 8300, 16.0: 8300, 16.5: 8300, 17.0: 8300, 17.5: 8300, 18.0: 8300, 18.5: 8300,
                  19.0: 8300, 19.5: 8300, 20.0: 8300, 20.5: 8300, 21.0: 8300, 21.5: 8300, 22.0: 8300, 22.5: 8300,
                  23.0: 8300, 23.5: 8300, 24.0: 8300, 24.5: 8300, 25.0: 8300}

def get_max_power(curve, ws):
    speeds = sorted(curve.keys())
    if ws < speeds[0]:
        return 0.0
    if ws >= speeds[-1]:
        return float(curve[speeds[-1]])
    for i in range(len(speeds) - 1):
        if speeds[i] <= ws < speeds[i + 1]:
            frac = (ws - speeds[i]) / (speeds[i + 1] - speeds[i])
            return curve[speeds[i]] + frac * (curve[speeds[i + 1]] - curve[speeds[i]])
    return float(curve[speeds[-1]])

# ==================== 读取数据 ====================
# ==================== 读取数据 ====================
def load_data():
    paths = [
        "/新新ps_1#_6.8MW_6step.csv",
        "/新新ps_4#_8.3MW6step.csv",
        "/新新ps_8#_6MW6step.csv",
        "/新新ps_12#_8MW6step.csv",
        "/新新ps_13#_8MW6step.csv",
        "/新新ps_14#_6MW6step.csv",
        "/新新ps_60#_6.8MW6step.csv",
        "/新新ps_67#_8.3MW_6step.csv"
    ]
    types = ['6.8', '6.8', '6.8', '6.8', '8.3', '8.3', '8.3', '8.3']
    curves = {'6.8': power_curve_68, '8.3': power_curve_83}
    startup_costs = {'6.8': 8.0, '8.3': 10.0}

    dfs = []
    for path in paths:
        try:
            df = pd.read_csv(path)
        except:
            df = pd.read_csv(path, encoding='gbk')
        df = df[['平均修正风速','平均电网有功功率']].copy()
        dfs.append(df)

    # 找到最短的长度
    min_len = min([len(df) for df in dfs])

    # 裁剪到最短长度
    dfs = [df.iloc[:min_len] for df in dfs]

    # wind_speeds
    wind_speeds = [df['平均修正风速'].values.astype(float) for df in dfs]

    # demands = 每行求和
    active_power_df = pd.concat([df['平均电网有功功率'] for df in dfs], axis=1)
    active_power_df = active_power_df.iloc[:min_len]  # 裁剪一致
    demands = active_power_df.sum(axis=1).values

    return wind_speeds, demands, types, curves, startup_costs

# ==================== 环境 ====================
class WindEnv:
    def __init__(self, wind_speeds, demands, types, curves, startup_costs):
        self.wind_speeds = wind_speeds
        self.demands = demands
        self.types = types
        self.curves = curves
        self.startup_costs = startup_costs
        self.num_turbines = 8
        self.max_steps = len(demands)
        self.rated = np.array([6800.0 if t == '6.8' else 8300.0 for t in types])
        self.total_rated = np.sum(self.rated)
        self.reset()

    def reset(self):
        self.current_step = 0
        self.previous_p = np.zeros(self.num_turbines)
        self.on_hours = np.zeros(self.num_turbines, dtype=int)
        self.off_hours = np.zeros(self.num_turbines, dtype=int)
        self.forced_state = np.array(["free"] * self.num_turbines)
        return self._get_state()

    def _get_state(self):
        ws = [self.wind_speeds[i][self.current_step] for i in range(self.num_turbines)]
        max_p = np.array([get_max_power(self.curves[self.types[i]], ws[i]) for i in range(self.num_turbines)])
        state = np.concatenate([
            self.previous_p / self.rated,
            max_p / self.rated,
            [self.demands[self.current_step] / self.total_rated],
            self.on_hours / 10.0,
            self.off_hours / 10.0
        ]).astype(np.float32)
        return state

    def step(self, action):
        ws = [self.wind_speeds[i][self.current_step] for i in range(self.num_turbines)]
        max_p = np.array([get_max_power(self.curves[self.types[i]], ws[i]) for i in range(self.num_turbines)])

        baseline = self.demands[self.current_step] * max_p / (np.sum(max_p) + 1e-6)
        raw_action = baseline + action * max_p * 0.5
        current_p = np.clip(raw_action, 0.0, max_p)

        sum_p = np.sum(current_p)
        demand = self.demands[self.current_step]

        # ===== 修改 reward =====
        deviation_term = abs(sum_p - demand) * 0.11
        change_term = np.sum(np.abs(current_p - self.previous_p)) * 0.0015
        reward = -(deviation_term + change_term)
        # =====================

        self.previous_p = current_p.copy()
        self.current_step += 1
        done = self.current_step >= self.max_steps
        next_state = np.zeros(33, dtype=np.float32) if done else self._get_state()

        info = {"sum_power": sum_p, "demand": demand}
        return next_state, reward, done, info


# ==================== TD3 网络 ====================
class Actor(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 512), nn.ReLU(),
            nn.Linear(512, 512), nn.ReLU(),
            nn.Linear(512, action_dim),
            nn.Tanh()
        )
    def forward(self, x):
        return self.net(x)

class Critic(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim + action_dim, 512), nn.ReLU(),
            nn.Linear(512, 512), nn.ReLU(),
            nn.Linear(512, 1)
        )
    def forward(self, s, a):
        return self.net(torch.cat([s, a], 1))

class TD3:
    def __init__(self, state_dim, action_dim, device):
        self.device = torch.device(device)
        self.actor = Actor(state_dim, action_dim).to(self.device)
        self.actor_target = copy.deepcopy(self.actor)
        self.critic1 = Critic(state_dim, action_dim).to(self.device)
        self.critic2 = Critic(state_dim, action_dim).to(self.device)
        self.critic_target1 = copy.deepcopy(self.critic1)
        self.critic_target2 = copy.deepcopy(self.critic2)
        self.actor_opt = optim.Adam(self.actor.parameters(), lr=5e-4)
        self.critic_opt = optim.Adam(list(self.critic1.parameters()) + list(self.critic2.parameters()), lr=5e-4)
        self.buffer = deque(maxlen=500000)
        self.batch_size = 512
        self.gamma = 0.99
        self.tau = 0.005
        self.policy_freq = 2
        self.total_it = 0

    def select_action(self, state, noise=0.1):
        state = torch.FloatTensor(state).unsqueeze(0).to(self.device)
        a = self.actor(state).cpu().detach().numpy()[0]
        if noise > 0:
            a = np.clip(a + np.random.normal(0, noise, size=a.shape), -1, 1)
        return a

    def store(self, s, a, s_, r, d):
        self.buffer.append((s, a, s_, r, float(d)))

    def update(self):
        if len(self.buffer) < self.batch_size:
            return
        self.total_it += 1
        batch = random.sample(self.buffer, self.batch_size)
        s, a, s_, r, d = map(np.array, zip(*batch))
        s = torch.FloatTensor(s).to(self.device)
        a = torch.FloatTensor(a).to(self.device)
        s_ = torch.FloatTensor(s_).to(self.device)
        r = torch.FloatTensor(r).unsqueeze(1).to(self.device)
        d = torch.FloatTensor(d).unsqueeze(1).to(self.device)

        with torch.no_grad():
            q1 = self.critic_target1(s_, self.actor_target(s_))
            q2 = self.critic_target2(s_, self.actor_target(s_))
            y = r + self.gamma * (1 - d) * torch.min(q1, q2)

        loss = F.mse_loss(self.critic1(s, a), y) + F.mse_loss(self.critic2(s, a), y)
        self.critic_opt.zero_grad()
        loss.backward()
        self.critic_opt.step()

        if self.total_it % self.policy_freq == 0:
            actor_loss = -self.critic1(s, self.actor(s)).mean()
            self.actor_opt.zero_grad()
            actor_loss.backward()
            self.actor_opt.step()

            for p, tp in zip(self.actor.parameters(), self.actor_target.parameters()):
                tp.data.copy_(self.tau * p.data + (1 - self.tau) * tp.data)

# ==================== 训练 ====================
# ==================== 训练 ====================
def train_td3(env, agent, episodes=500):
    rewards = []
    save_dir = "D:/风电/数据收集/"
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, "td3_episode_rewards_ε=0.5.csv")

    for ep in range(episodes):
        s = env.reset()
        done = False
        ep_reward = 0
        noise = 0.5 * (0.995 ** ep)
        while not done:
            a = agent.select_action(s, noise)
            s_, r, done, _ = env.step(a)
            agent.store(s, a, s_, r, done)
            agent.update()
            s = s_
            ep_reward += r
        rewards.append(ep_reward)

        # 每 10 回合保存一次 reward
        if (ep + 1) % 10 == 0:
            pd.DataFrame({
                "episode": np.arange(1, len(rewards) + 1),
                "reward": rewards
            }).to_csv(save_path, index=False, encoding="utf-8-sig")

        if (ep + 1) % 100 == 0:
            print(f"Episode {ep+1:4d} | Reward {ep_reward:8.3f}")

    return rewards


# ==================== 主程序 ====================
if __name__ == "__main__":
    seed = 42
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    wind_speeds, demands, types, curves, startup_costs = load_data()
    env = WindEnv(wind_speeds, demands, types, curves, startup_costs)

    agent = TD3(state_dim=33, action_dim=8,
                device='cuda' if torch.cuda.is_available() else 'cpu')

    rewards = train_td3(env, agent, episodes=500)

     #======== 保存 reward 到 CSV ========
    save_dir = "D:/风电/TD3+动作投影+残差动作数据收集/"
    os.makedirs(save_dir, exist_ok=True)

    save_path = os.path.join(save_dir, "td3_episode_rewards10_ε=0.5.csv")
    pd.DataFrame({
        "episode": np.arange(1, len(rewards) + 1),
        "reward": rewards
    }).to_csv(save_path, index=False, encoding="utf-8-sig")

    print(f"\n✅ 每回合 reward 已保存至：{save_path}")
